In [1]:
%reload_ext autoreload
%autoreload 2

import pandas as pd
pd.options.plotting.backend = "plotly"

import sys
sys.path.insert(0, 'driftpy/src/')

import driftpy
import numpy as np 
from sim.sim import SimpleDriftSim, load_hist_oracle

import os
import datetime
from sim.agents import * 
from programs.clearing_house.state import * 
from sim.events import OpenPositionEvent
from sim.helpers import random_walk_oracle, rand_heterosk_oracle, class_to_json
import pickle as cPickle

# initial setup 
- setup the agents in the simulation 
- setup the market/clearing house 

In [21]:
def make_clearing_house(
    base_spread, 
    oracle
):
    # create the market with/without spread 
    amm = AMM(
        oracle=oracle, 
        base_asset_reserve=int(10_000_000 * AMM_RESERVE_PRECISION), 
        quote_asset_reserve=int(10_000_000 * AMM_RESERVE_PRECISION),
        funding_period=60*60,
        peg_multiplier=int(oracle.get_price(0) * PEG_PRECISION),
        base_spread=base_spread
    )
    market = Market(amm)
        
    # create clearing house 
    fee_structure = FeeStructure(numerator=1, denominator=1000)
    clearing_house = ClearingHouse([market], fee_structure)
    clearing_house.name = f''

    return clearing_house

prices, timestamps = random_walk_oracle(1, n_steps=20) # price follows a random walk 
oracle = Oracle(prices=prices, timestamps=timestamps)
len(prices)

20

In [22]:
# agent which always pushes mark => oracle 
arb = Arb(
    intensity=0.9, 
    market_index=0, 
    user_index=0,
)

# random trader 
noise = Noise(
    intensity=1, 
    market_index=0, 
    user_index=0,
    size=1_000, # 1k trades
)

agents = [arb, noise]

## simulation with no spread (base_spread=0)

In [23]:
os.makedirs("sim-results/", exist_ok=True)

no_spread_clearing_house = make_clearing_house(base_spread=0, oracle=oracle)

# without spread sim 
no_spread_sim = SimpleDriftSim(
    "sim-results/no_spread",
    clearing_house=no_spread_clearing_house,
    agents=agents
)

no_spread_sim.run()
no_spread_result_df = no_spread_sim.to_df(save=True).add_prefix('noSpread_')
no_spread_result_df.head()

running simulation for 95 timesteps
running sim from timestamp 0 to 95
skipping time step: 0 ... ch time: 3
skipping time step: 1 ... ch time: 3
skipping time step: 2 ... ch time: 3
ERR: <class 'sumtypes.SHORT'> 1.0010532424511382 1.0010532424511382 1.0010532424 1.001049504146912
ERR: <class 'sumtypes.SHORT'> 4.4163124397560525 4.4163124397560525 4.4163124397 4.4162395525866085
ERR: <class 'sumtypes.SHORT'> 4.4167136000657266 4.4167136000657266 4.4167136 4.4166970761008795
ERR: <class 'sumtypes.SHORT'> 1.7654816309243806 1.7654816309243806 1.7654816309 1.765477446437169
ERR: <class 'sumtypes.SHORT'> 2.6370641886563435 2.6370641886563435 2.6370641886 2.637063276997314
saving results to sim-results/no_spread...


100%|██████████| 187/187 [00:00<00:00, 4292.25it/s]


,noSpread_m0_mark_price,noSpread_m0_oracle_price,noSpread_m0_bid_price,noSpread_m0_ask_price,noSpread_m0_wouldbe_peg,noSpread_m0_wouldbe_peg_cost,noSpread_m0_predicted_long_funding,noSpread_m0_predicted_short_funding,noSpread_m0_last_mid_price_twap,noSpread_m0_repeg_to_oracle_cost,...,noSpread_u1_total_fee_rebate,noSpread_u1_open_orders,noSpread_u1_cumulative_deposits,noSpread_u1_total_collateral,noSpread_u0_m0_base_asset_amount,noSpread_u0_m0_quote_asset_amount,noSpread_u0_m0_last_cumulative_funding_rate,noSpread_u0_m0_last_funding_rate_ts,noSpread_u0_m0_upnl,noSpread_u0_m0_ufunding
0,4.3910,4.391993,4.3910,4.3910,4.391,0.0,-9.425155e-08,-9.425155e-08,4.391,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4.3910,4.391993,4.3910,4.3910,4.391,0.0,-9.425155e-08,-9.425155e-08,4.391,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.3910,4.391993,4.3910,4.3910,4.391,0.0,-9.425155e-08,-9.425155e-08,4.391,0.0,...,0.0,0.0,1.000000e+13,1.000000e+13,NaN,NaN,NaN,NaN,NaN,NaN
3,4.3910,4.391993,4.3910,4.3910,4.391,0.0,-9.425155e-08,-9.425155e-08,4.391,0.0,...,0.0,0.0,1.000000e+13,1.000000e+13,NaN,NaN,NaN,NaN,NaN,NaN
4,4.3908,4.391993,4.3908,4.3908,4.391,0.0,-9.425155e-08,-9.425155e-08,4.391,0.0,...,0.0,0.0,1.000000e+13,1.000000e+13,-2.277437e+15,1.000000e+09,0.0,0.0,-0.99964,-0.0


In [24]:
no_spread_result_df[[
    'noSpread_m0_mark_price',
    'noSpread_m0_oracle_price',
]].plot()

In [25]:
results = no_spread_result_df
keep_columns = results.columns
keep_columns = [c for c in keep_columns if results[c].dtype == float or results[c].dtype == int]
results[keep_columns].plot()

In [26]:
spread_values = pd.DataFrame(
    data={'spread': (
        no_spread_result_df[['noSpread_m0_ask_price']].values
        - no_spread_result_df[['noSpread_m0_bid_price']].values 
    ).flatten()}
)
spread_values.plot(title='base spread = 0: spread')

# simulation with a spread (base_spread=100)

In [27]:
# run again with spread sim 
spread_clearing_house = make_clearing_house(base_spread=100, oracle=oracle)

spread_sim = SimpleDriftSim(
    "sim-results/spread",
    clearing_house=spread_clearing_house,
    agents=agents
)

spread_sim.run()
spread_result_df = spread_sim.to_df(save=True).add_prefix('withSpread_')
spread_result_df.head()

running simulation for 95 timesteps
running sim from timestamp 0 to 95
skipping time step: 0 ... ch time: 3
skipping time step: 1 ... ch time: 3
skipping time step: 2 ... ch time: 3
saving results to sim-results/spread...


100%|██████████| 187/187 [00:00<00:00, 4309.86it/s]


,withSpread_m0_mark_price,withSpread_m0_oracle_price,withSpread_m0_bid_price,withSpread_m0_ask_price,withSpread_m0_wouldbe_peg,withSpread_m0_wouldbe_peg_cost,withSpread_m0_predicted_long_funding,withSpread_m0_predicted_short_funding,withSpread_m0_last_mid_price_twap,withSpread_m0_repeg_to_oracle_cost,...,withSpread_u1_total_fee_rebate,withSpread_u1_open_orders,withSpread_u1_cumulative_deposits,withSpread_u1_total_collateral,withSpread_u0_m0_base_asset_amount,withSpread_u0_m0_quote_asset_amount,withSpread_u0_m0_last_cumulative_funding_rate,withSpread_u0_m0_last_funding_rate_ts,withSpread_u0_m0_upnl,withSpread_u0_m0_ufunding
0,4.3910,4.391993,4.39078,4.39122,4.391,0.0,-9.425155e-08,-9.425155e-08,4.391,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4.3910,4.391993,4.39078,4.39122,4.391,0.0,-9.425155e-08,-9.425155e-08,4.391,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.3910,4.391993,4.39078,4.39122,4.391,0.0,-9.425155e-08,-9.425155e-08,4.391,0.0,...,0.0,0.0,1.000000e+13,1.000000e+13,NaN,NaN,NaN,NaN,NaN,NaN
3,4.3910,4.391993,4.39078,4.39122,4.391,0.0,-9.425155e-08,-9.425155e-08,4.391,0.0,...,0.0,0.0,1.000000e+13,1.000000e+13,NaN,NaN,NaN,NaN,NaN,NaN
4,4.3908,4.391993,4.39058,4.39102,4.391,0.0,-9.425155e-08,-9.425155e-08,4.391,0.0,...,0.0,0.0,1.000000e+13,1.000000e+13,-2.277551e+15,1.000000e+09,0.0,0.0,-50004.573876,-0.0


In [28]:
spread_df = pd.DataFrame(
    data={'spread': (
        spread_result_df[['withSpread_m0_ask_price']].values 
        - spread_result_df[['withSpread_m0_bid_price']].values 
    ).flatten()}
)
spread_df.plot(title='base spread = 100: spread')